In [32]:
import pandas as pd
df = pd.read_excel('/home/stepan/work/MARVEL/marvel_characters.xlsx')
df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

/tmp/ipykernel_3299190/975869265.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [36]:
for column in df.columns:
    unique_values = df[column].unique()
    if len(unique_values) < 10:
        print(f"Колонка '{column}': {unique_values}")
        print(f"Количество уникальных значений: {len(unique_values)}\n")

Колонка 'role': ['супергерой' 'антигерой' 'злодей' 'антагонист']
Количество уникальных значений: 4

Колонка 'status': ['погиб' 'жив' 'погибла' 'жива' 'уничтожен']
Количество уникальных значений: 5



In [38]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine

In [39]:
engine = create_engine("sqlite:////home/stepan/work/MARVEL/MarvelLLM/data/marvel_characters.db")
df.to_sql("marvel", engine, index=False)

51

---

In [40]:
import sqlite3

db_path = '/home/stepan/work/MARVEL/MarvelLLM/data/marvel_characters.db'

conn = sqlite3.connect(db_path)

In [41]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Список таблиц:", tables)

Список таблиц:      name
0  marvel


In [42]:
first_table = tables.iloc[0, 0]
df = pd.read_sql(f"SELECT * FROM {first_table} LIMIT 10;", conn)

In [43]:
df

,character,real_name,affiliation,role,powers,powers_origin,species,status,first_appearance
0,железный человек,тони старк,мстители,супергерой,"интеллект, бронекостюм",технологии,человек,погиб,tales of suspense #39
1,капитан америка,стив роджерс,мстители,супергерой,"сила, тактика",сыворотка суперсолдата,человек,жив,captain america comics #1
2,тор,тор одинсон,"мстители, асгард",супергерой,"молнии, божественная сила",бог,асгардец,жив,journey into mystery #83
3,халк,брюс бэннер,мстители,супергерой,"сверхсила, регенерация",гамма-радиация,человек,жив,the incredible hulk #1
4,чёрная вдова,наташа романофф,мстители,супергерой,"боевые искусства, шпионаж",подготовка,человек,погибла,tales of suspense #52
5,соколиный глаз,клинт бартон,мстители,супергерой,меткость,тренировки,человек,жив,tales of suspense #57
6,вижн,вижн,мстители,супергерой,"фазирование, интеллект",ии,синтетик,погиб,avengers #57
7,алая ведьма,ванда максимофф,мстители,супергерой,магия хаоса,"мутация, магия",мутант,жива,x-men #4
8,ртуть,пьетро максимофф,мстители,супергерой,сверхскорость,мутация,мутант,погиб,x-men #4
9,капитан марвел,кэрол дэнверс,мстители,супергерой,космическая энергия,кри-взрыв,человек,жива,marvel super-heroes #13


In [46]:
query = """
SELECT character, affiliation, role, powers_origin, species
FROM marvel
WHERE role LIKE '%супергерой%'
  AND affiliation LIKE '%мстители%'
  AND powers_origin LIKE '%технологии%'
LIMIT 10;
"""

df = pd.read_sql(query, conn)
df

,character,affiliation,role,powers_origin,species
0,железный человек,мстители,супергерой,технологии,человек
